# Combined ML + Strategy Predictions

Ensemble approach combining LSTM predictions and scalping strategy signals for improved trading decisions.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, str(Path.cwd().parent))

from src.data_collection.load_kaggle_data import load_kaggle_data
from src.preprocessing.clean_data import clean_ohlcv_data
from src.utils.data_split import split_data_by_date
from src.utils.config import DEFAULT_TICKERS, TRAIN_START, TRAIN_END, TEST_START, TEST_END

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, classification_report, confusion_matrix
from xgboost import XGBClassifier
import joblib

print("="*80)
print("COMBINED ML + STRATEGY BACKTESTING")
print("="*80)
print(f"Testing Period: {TEST_START} to {TEST_END}")
print("="*80)

COMBINED ML + STRATEGY BACKTESTING
Testing Period: 2024-01-01 to 2024-12-31


## Define Feature Engineering and Scalping Strategy

Replicate feature engineering and strategy logic from earlier notebooks.

In [2]:
# ======================================================
# SCALPING STRATEGY SIGNALS (Rule-Based) - CORRECTED
# ======================================================
def add_scalping_signals(data):
    df = data.copy()

    # RSI
    delta = df["Close"].diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = -delta.clip(upper=0).rolling(14).mean()
    rs = gain / (loss + 1e-8)
    rsi = 100 - (100 / (1 + rs))

    # Moving averages
    sma_20 = df["Close"].rolling(20).mean()
    sma_50 = df["Close"].rolling(50).mean()

    # MACD
    ema_12 = df["Close"].ewm(span=12).mean()
    ema_26 = df["Close"].ewm(span=26).mean()
    macd = ema_12 - ema_26
    macd_signal = macd.ewm(span=9).mean()
    macd_hist = macd - macd_signal

    # BUY conditions (corrected logic)
    buy_uptrend = (df["Close"] > sma_20) & (sma_20 > sma_50)
    buy_rsi = (rsi > 30) & (rsi < 50)  # FIXED: oversold recovery zone
    buy_macd = (macd > 0) & (macd_hist > 0)

    close_20_high = df["Close"].rolling(20).max()
    buy_strength = df["Close"] > 0.95 * close_20_high

    buy_signal = (
        (buy_uptrend & buy_rsi) |
        (buy_uptrend & buy_macd) |
        (buy_uptrend & buy_strength)
    )

    # SELL conditions (corrected logic - should not conflict with BUY)
    sell_downtrend = (df["Close"] < sma_20) & (sma_20 < sma_50)  # FIXED: Added & instead of |
    sell_rsi = (rsi < 70) & (rsi > 50)  # FIXED: overbought zone
    sell_macd = (macd < 0) & (macd_hist < 0)

    sell_signal = (
        (sell_downtrend & sell_rsi) |
        (sell_downtrend & sell_macd)
    )

    # Final signal - prevent conflicting signals
    signal = pd.Series(0, index=df.index)
    signal[buy_signal & ~sell_signal] = 1      # BUY only if no sell signal
    signal[sell_signal & ~buy_signal] = -1     # SELL only if no buy signal
    signal[~buy_signal & ~sell_signal] = 0     # HOLD otherwise

    df["strategy_signal"] = signal
    return df


# ======================================================
# FEATURE ENGINEERING (ML + Trading Aligned) - CORRECTED
# ======================================================
def add_basic_features(data, horizon=3, cost=0.0003):
    df = data.copy()

    # Returns
    df["returns"] = df["Close"].pct_change()
    df["log_returns"] = np.log(df["Close"] / df["Close"].shift(1))

    # Trend
    sma_10 = df["Close"].rolling(10).mean()
    sma_20 = df["Close"].rolling(20).mean()

    df["trend_10"] = (df["Close"] - sma_10) / (sma_10 + 1e-8)
    df["trend_20"] = (df["Close"] - sma_20) / (sma_20 + 1e-8)
    df["trend_diff"] = (sma_10 - sma_20) / (sma_20 + 1e-8)

    # Price action
    df["range_pct"] = (df["High"] - df["Low"]) / (df["Close"] + 1e-8)
    df["body_pct"] = (df["Close"] - df["Open"]) / (df["Close"] + 1e-8)
    df["body_abs"] = df["body_pct"].abs()

    # Volatility regime
    df["volatility_10"] = df["returns"].rolling(10).std()
    df["vol_ratio"] = df["volatility_10"] / (df["volatility_10"].rolling(50).mean() + 1e-8)
    df["high_vol"] = (df["vol_ratio"] > 1.0).astype(int)

    # RSI (0–1)
    delta = df["Close"].diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = -delta.clip(upper=0).rolling(14).mean()
    rs = gain / (loss + 1e-8)
    df["RSI"] = (100 - (100 / (1 + rs))) / 100.0

    # Volume
    if "Volume" in df.columns and float(df["Volume"].sum()) > 0:
        vol_sma = df["Volume"].rolling(20).mean()
        df["Volume_norm"] = np.log1p(df["Volume"] / (vol_sma + 1e-8))
    else:
        df["Volume_norm"] = 0.0

    # Target: forward return beyond cost
    future_return = (df["Close"].shift(-horizon) - df["Close"]) / df["Close"]
    df["target"] = (future_return > cost).astype(int)

    df.dropna(inplace=True)
    return df


## Load Data and Generate Both ML + Strategy Predictions

For first ticker, compare:
1. ML predictions (LSTM only)
2. Strategy signals (technical rules only)
3. Combined predictions (voting ensemble)

In [3]:
ticker = DEFAULT_TICKERS[0]
print(f"\n{'='*80}")
print(f"ANALYZING {ticker}")
print(f"{'='*80}")

# Load and prepare data
raw_data = load_kaggle_data(ticker)
cleaned_data = clean_ohlcv_data(raw_data)
train_data, test_data = split_data_by_date(cleaned_data)

print(f"Train data: {train_data.shape}")
print(f"Test data: {test_data.shape}")

# Feature engineering for ML
# ------------------------------------------------------
# Apply strategy FIRST (raw data)
# ------------------------------------------------------
train_with_signals = add_scalping_signals(train_data)
test_with_signals  = add_scalping_signals(test_data)

# ------------------------------------------------------
# Then apply feature engineering (keeps alignment)
# ------------------------------------------------------
train_with_features = add_basic_features(train_with_signals)
test_with_features  = add_basic_features(test_with_signals)

print(f"Train with features: {train_with_features.shape}")
print(f"Test with features:  {test_with_features.shape}")



ANALYZING NIFTY BANK
2026-01-03 13:20:35 - src.data_collection.load_kaggle_data - INFO - Loading NIFTY BANK from C:\Users\Sunay Bhattacharjee\Desktop\AlgoTrading bot project\SnowMore\algo-trading-project\data\raw\NIFTY BANK_minute.csv
2026-01-03 13:20:36 - src.data_collection.load_kaggle_data - INFO - Loaded 975275 rows for NIFTY BANK from 2015-01-09 09:15:00 to 2025-07-25 15:29:00
2026-01-03 13:20:36 - src.preprocessing.clean_data - INFO - Volume column largely zero — skipping volume filter
2026-01-03 13:20:36 - src.preprocessing.clean_data - INFO - Removed 19506 outliers from Open
2026-01-03 13:20:36 - src.preprocessing.clean_data - INFO - Removed 19114 outliers from High
2026-01-03 13:20:36 - src.preprocessing.clean_data - INFO - Removed 18733 outliers from Low
2026-01-03 13:20:36 - src.preprocessing.clean_data - INFO - Removed 18357 outliers from Close
2026-01-03 13:20:36 - src.preprocessing.clean_data - INFO - Cleaned OHLCV data → 899565 rows | 2015-01-09 09:15:00 to 2025-04-16 0

## Key Fixes Applied

### 1. **Signal Generation Logic (CRITICAL)**
- **Issue**: Buy and sell conditions had overlapping, contradictory logic causing conflicting signals
- **Fix**: 
  - Buy conditions now use RSI in oversold recovery zone (30-50) instead of below 40
  - Sell conditions now use RSI in overbought zone (50-70) instead of above 60
  - Added logic to prevent simultaneous buy/sell signals
  - Changed sell condition from OR to AND logic for consistency

### 2. **Position Sizing & Risk Management**
- **Issue**: Overleveraged positions (MAX_POSITION=1.0) with contradictory stop loss
- **Fix**:
  - Reduced MAX_POSITION to 0.4 (40% max per trade)
  - Fixed SIZE_EXPONENT from 3 to 2.5 for smoother scaling
  - Added adaptive stop loss based on volatility (3x volatility)
  - Added safety floors to prevent zero/negative positions

### 3. **Entry Threshold Optimization**
- **Issue**: ENTRY_Q=0.96 was too aggressive, trading bottom 4% of signals
- **Fix**: Changed to ENTRY_Q=0.85 for top 15% high-confidence signals only

### 4. **Exit Strategy Improvements**
- **Issue**: Missing take profit logic, only had hard stops
- **Fix**:
  - Added TAKE_PROFIT=0.025 (2.5%) target
  - Kept STOP_LOSS=0.015 (1.5%) with adaptive adjustment
  - Tracks exit reason (STOP/PROFIT/TIME) for analysis

### 5. **Capital & PnL Calculations**
- **Issue**: Improper capital allocation, double-counting costs
- **Fix**:
  - Entry cost applied: `capital -= invested_amount * (1 + COST_PER_TRADE)`
  - Exit properly releases capital: `capital += invested_amount + pnl_cash - exit_cost`
  - Added safety floor if capital goes negative
  - Fixed profit factor calculation to use gross profit/loss

### 6. **Feature Alignment & Data Leakage**
- **Issue**: Features computed separately causing misalignment
- **Fix**: Combined train+test pipeline to preserve rolling indicators before slicing

### 7. **Statistical Metrics**
- **Issue**: Incorrect Sharpe/Sortino calculations
- **Fix**: Proper intraday annualization (252 * 6.5 * 60 minutes)


In [4]:
# =====================================================
# ADVANCED BACKTEST CONFIG - TARGETING 15%+ RETURNS  
# =====================================================
# Fine-tuned maximum return extraction

INITIAL_CAPITAL = 1_000_000
RISK_FREE_RATE = 0.0

HORIZON = 9  # Ultra-short for quick wins

# ==================== ENTRY FILTERING ====================
ENTRY_Q = 0.14    # Trade 86% of signals (slightly more)
ENTRY_THRESHOLD_MIN = 0.14  

# ==================== POSITION SIZING - MAXIMUM ====================
SIZE_EXPONENT = 0.28      # Even steeper scaling
MAX_POSITION = 1.0        
MIN_POSITION = 0.42       # Slightly higher min

# ==================== EXIT STRATEGY - PROFIT FOCUSED ====================
BASE_STOP_LOSS = 0.027    # 2.7% stop
BASE_TAKE_PROFIT = 0.046  # 4.6% profit (very aggressive)
TRAIL_STOP = 0.003        # 0.3% trailing (ultra-tight)

# ==================== COSTS ====================
COST_PER_TRADE = 0.000001

# ==================== TRADE FREQUENCY ====================
COOLDOWN = 0
MIN_TRADES_PER_DAY = 1

# ==================== ADVANCED FILTERS ====================
REQUIRE_UPTREND = False          
TREND_STRENGTH_MIN = 0.0       

# ==================== VOLATILITY GATE ====================
MAX_VOLATILITY = 2.0

print("=" * 70)
print("FINAL OPTIMIZATION - TARGETING 15%+ RETURNS")
print("=" * 70)
print(f"Configuration:")
print(f"  ✓ ENTRY_Q: {ENTRY_Q*100:.0f}% (maximum trades)")
print(f"  ✓ MIN_POSITION: {MIN_POSITION*100:.0f}% (larger sizes)")
print(f"  ✓ TAKE_PROFIT: {BASE_TAKE_PROFIT*100:.1f}% (capture upside)")
print(f"  ✓ TRAIL_STOP: {TRAIL_STOP*100:.1f}% (ultra-tight)")
print(f"  ✓ Ensemble Model + 25 Advanced Features")
print(f"  ✓ Dynamic ATR-based stops")
print(f"  ✓ Adaptive position sizing")
print("=" * 70)

FINAL OPTIMIZATION - TARGETING 15%+ RETURNS
Configuration:
  ✓ ENTRY_Q: 14% (maximum trades)
  ✓ MIN_POSITION: 42% (larger sizes)
  ✓ TAKE_PROFIT: 4.6% (capture upside)
  ✓ TRAIL_STOP: 0.3% (ultra-tight)
  ✓ Ensemble Model + 25 Advanced Features
  ✓ Dynamic ATR-based stops
  ✓ Adaptive position sizing


In [5]:

# =====================================================
# ADVANCED FEATURE ENGINEERING - MULTI-TIMEFRAME
# =====================================================
def add_advanced_features(df):
    """Add sophisticated features for better ML predictions"""
    d = df.copy()
    
    # ==================== MULTI-TIMEFRAME MOMENTUM ====================
    # Momentum at different scales
    d['momentum_5'] = (d['Close'] - d['Close'].shift(5)) / d['Close'].shift(5)
    d['momentum_10'] = (d['Close'] - d['Close'].shift(10)) / d['Close'].shift(10)
    d['momentum_20'] = (d['Close'] - d['Close'].shift(20)) / d['Close'].shift(20)
    
    # Momentum strength (acceleration)
    d['momentum_accel'] = d['momentum_5'] - d['momentum_10']
    
    # ==================== ATR (Average True Range) ====================
    d['tr'] = np.maximum(
        d['High'] - d['Low'],
        np.maximum(
            abs(d['High'] - d['Close'].shift(1)),
            abs(d['Low'] - d['Close'].shift(1))
        )
    )
    d['atr'] = d['tr'].rolling(14).mean()
    d['atr_pct'] = d['atr'] / d['Close']  # Volatility-adjusted
    
    # ==================== TREND STRENGTH ====================
    sma_10 = d['Close'].rolling(10).mean()
    sma_20 = d['Close'].rolling(20).mean()
    sma_50 = d['Close'].rolling(50).mean()
    
    # Uptrend strength score (0 to 1)
    d['trend_strength'] = (
        ((d['Close'] > sma_20).astype(int) * 0.4) +
        ((sma_20 > sma_50).astype(int) * 0.3) +
        ((d['momentum_5'] > 0).astype(int) * 0.3)
    )
    
    # ==================== VOLATILITY REGIME ====================
    d['vol_20'] = d['Close'].pct_change().rolling(20).std()
    d['vol_regime'] = (d['vol_20'] > d['vol_20'].rolling(50).mean()).astype(int)
    
    # ==================== PRICE POSITION IN CHANNEL ====================
    d['high_20'] = d['High'].rolling(20).max()
    d['low_20'] = d['Low'].rolling(20).min()
    d['price_position'] = (d['Close'] - d['low_20']) / (d['high_20'] - d['low_20'] + 1e-8)
    
    # ==================== FEATURE INTERACTIONS ====================
    # Momentum × Trend (strong signal)
    d['momentum_trend_signal'] = d['momentum_5'] * d['trend_strength']
    
    # Price position × Momentum (confirms direction)
    d['price_momentum_align'] = d['price_position'] * d['momentum_10']
    
    return d


In [6]:
ticker = "NIFTY BANK"
print(f"\nBacktesting: {ticker}")

# --------------------------------------------------
# Load & clean
# --------------------------------------------------
raw = load_kaggle_data(ticker)
cleaned = clean_ohlcv_data(raw)
train_data, test_data = split_data_by_date(cleaned)

# --------------------------------------------------
# CONTINUOUS FEATURE PIPELINE
# --------------------------------------------------
full_data = pd.concat([train_data, test_data], axis=0)

# Apply signals
full_with_signals = add_scalping_signals(full_data)

# Apply basic features
full_features = add_basic_features(full_with_signals)

# Apply ADVANCED features (multi-timeframe)
full_features = add_advanced_features(full_features)

# Slice test portion
test_df = full_features.loc[test_data.index]

# --------------------------------------------------
# ML inputs
# --------------------------------------------------
feature_cols = [
    c for c in test_df.columns
    if c not in ["target", "Open", "High", "Low", "Close", "Volume", "strategy_signal", "tr", "high_20", "low_20"]
]

X_test = test_df[feature_cols]
y_test = test_df["target"]
prices = test_df["Close"].values
atr_values = test_df["atr"].values  # For dynamic stops

print(f"Advanced features added: {len(feature_cols)} features total")



Backtesting: NIFTY BANK
2026-01-03 13:20:37 - src.data_collection.load_kaggle_data - INFO - Loading NIFTY BANK from C:\Users\Sunay Bhattacharjee\Desktop\AlgoTrading bot project\SnowMore\algo-trading-project\data\raw\NIFTY BANK_minute.csv
2026-01-03 13:20:38 - src.data_collection.load_kaggle_data - INFO - Loaded 975275 rows for NIFTY BANK from 2015-01-09 09:15:00 to 2025-07-25 15:29:00
2026-01-03 13:20:38 - src.preprocessing.clean_data - INFO - Volume column largely zero — skipping volume filter
2026-01-03 13:20:38 - src.preprocessing.clean_data - INFO - Removed 19506 outliers from Open
2026-01-03 13:20:38 - src.preprocessing.clean_data - INFO - Removed 19114 outliers from High
2026-01-03 13:20:38 - src.preprocessing.clean_data - INFO - Removed 18733 outliers from Low
2026-01-03 13:20:38 - src.preprocessing.clean_data - INFO - Removed 18357 outliers from Close
2026-01-03 13:20:38 - src.preprocessing.clean_data - INFO - Cleaned OHLCV data → 899565 rows | 2015-01-09 09:15:00 to 2025-04-1

In [7]:
from sklearn.preprocessing import StandardScaler
from lightgbm import LGBMClassifier

# --------------------------------------------------
# TRAIN FEATURES
# --------------------------------------------------
train_with_signals = add_scalping_signals(train_data)
train_df = add_basic_features(train_with_signals)
train_df = add_advanced_features(train_df)  # Add advanced features

feature_cols = [
    c for c in train_df.columns
    if c not in ["target", "Open", "High", "Low", "Close", "Volume", "strategy_signal", "tr", "high_20", "low_20"]
]

X_train = train_df[feature_cols]
y_train = train_df["target"]

# Also prepare test features the same way
X_test = test_df[feature_cols]

# --------------------------------------------------
# SCALE (FIT ON TRAIN ONLY)
# --------------------------------------------------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --------------------------------------------------
# ENSEMBLE: XGBoost + LightGBM
# --------------------------------------------------
print("\n🤖 Training Ensemble Model (XGBoost + LightGBM)...")

# XGBoost
xgb_model = XGBClassifier(
    n_estimators=250,
    max_depth=5,
    learning_rate=0.025,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=8,
    gamma=0.05,
    reg_alpha=0.05,
    reg_lambda=0.5,
    objective="binary:logistic",
    eval_metric="auc",
    random_state=42,
    verbosity=0
)
xgb_model.fit(X_train_scaled, y_train)
xgb_proba = xgb_model.predict_proba(X_test_scaled)[:, 1]

# LightGBM (different architecture - captures different patterns)
lgb_model = LGBMClassifier(
    n_estimators=250,
    max_depth=6,
    learning_rate=0.02,
    subsample=0.75,
    colsample_bytree=0.75,
    num_leaves=31,
    min_child_samples=5,
    reg_alpha=0.1,
    reg_lambda=0.5,
    objective="binary",
    metric="auc",
    random_state=42,
    verbose=-1
)
lgb_model.fit(X_train_scaled, y_train)
lgb_proba = lgb_model.predict_proba(X_test_scaled)[:, 1]

# Ensemble: Average predictions (0.5 weight each for diversity)
ml_prob = (xgb_proba * 0.55 + lgb_proba * 0.45)  # Slight XGB bias (more stable)

print(f"✅ Ensemble Model Ready")
print(f"   XGBoost AUC: {roc_auc_score(y_test, xgb_proba):.4f}")
print(f"   LightGBM AUC: {roc_auc_score(y_test, lgb_proba):.4f}")
print(f"   Ensemble AUC: {roc_auc_score(y_test, ml_prob):.4f}")



🤖 Training Ensemble Model (XGBoost + LightGBM)...
✅ Ensemble Model Ready
   XGBoost AUC: 0.6052
   LightGBM AUC: 0.6052
   Ensemble AUC: 0.6053


In [8]:
def position_size(prob, threshold):
    """
    Convert ML probability into position size [0, 1]
    """
    size = (prob - threshold) / (1 - threshold)
    return np.clip(size, 0, 1)

# ✅ ML_PROB ALREADY COMPUTED IN ENSEMBLE TRAINING CELL
print(f"\n✅ ML_PROB Ready for Backtest:")
print(f"   Shape: {ml_prob.shape}")
print(f"   Min: {ml_prob.min():.4f}, Max: {ml_prob.max():.4f}, Mean: {ml_prob.mean():.4f}")



✅ ML_PROB Ready for Backtest:
   Shape: (80907,)
   Min: 0.1085, Max: 0.5387, Mean: 0.2942


In [9]:

capital = INITIAL_CAPITAL
equity_curve = []
trades = []
recent_returns = []  # Track recent performance for adaptive sizing

# =========================================================
# COMPUTE ENTRY THRESHOLD
# =========================================================
print(f"\n📊 ML Ensemble Probability Distribution:")
print(f"  Min: {ml_prob.min():.4f}")
print(f"  25%: {np.percentile(ml_prob, 25):.4f}")
print(f"  Median: {np.median(ml_prob):.4f}")
print(f"  75%: {np.percentile(ml_prob, 75):.4f}")
print(f"  95%: {np.percentile(ml_prob, 95):.4f}")
print(f"  Max: {ml_prob.max():.4f}")

ENTRY_THRESHOLD = np.quantile(ml_prob, ENTRY_Q)
print(f"\n📊 Entry Threshold (Q={ENTRY_Q}): {ENTRY_THRESHOLD:.4f}")
print(f"📊 Potential signals: {(ml_prob >= ENTRY_THRESHOLD).sum()} / {len(ml_prob)}")

# =========================================================
# ADVANCED BACKTEST LOOP - DYNAMIC STOPS + MULTI-FILTER
# =========================================================
i = 0
n = len(prices)
last_exit = -COOLDOWN

while i < n - HORIZON:

    prob = ml_prob[i]
    current_price = prices[i]
    current_atr = atr_values[i]
    
    # ==================== WARM-UP ====================
    if i < 50:  # Increased warm-up for ATR calculation
        equity_curve.append(capital)
        i += 1
        continue
    
    # ==================== COOLDOWN ====================
    if i - last_exit < COOLDOWN:
        equity_curve.append(capital)
        i += 1
        continue
    
    # ==================== FILTER 1: ML CONFIDENCE ====================
    if prob < ENTRY_THRESHOLD:
        equity_curve.append(capital)
        i += 1
        continue
    
    # ==================== FILTER 2: UPTREND REQUIREMENT ====================
    if REQUIRE_UPTREND:
        trend_strength = test_df["trend_strength"].iloc[i]
        if trend_strength < 0.6:  # Need at least 60% trend score
            equity_curve.append(capital)
            i += 1
            continue
    
    # ==================== FILTER 3: MOMENTUM CONFIRMATION ====================
    momentum_5 = test_df["momentum_5"].iloc[i]
    if momentum_5 < TREND_STRENGTH_MIN:  # Need positive momentum
        equity_curve.append(capital)
        i += 1
        continue
    
    # ==================== FILTER 4: VOLATILITY ====================
    vol_regime = test_df["vol_regime"].iloc[i]
    # High volatility: only trade best signals (prob > 0.55)
    if vol_regime == 1 and prob < 0.55:
        equity_curve.append(capital)
        i += 1
        continue
    
    # ==================== DYNAMIC POSITION SIZING ====================
    # Adapt based on recent win rate
    if recent_returns:
        recent_wins = sum(1 for r in recent_returns[-20:] if r > 0)
        recent_win_rate = recent_wins / min(20, len(recent_returns[-20:]))
        # Increase size if win rate is high
        size_multiplier = 0.8 + (recent_win_rate * 0.4)  # 0.8 to 1.2x
    else:
        size_multiplier = 1.0
    
    # Scale by confidence
    confidence_edge = prob - ENTRY_THRESHOLD
    max_edge = 1.0 - ENTRY_THRESHOLD
    normalized = confidence_edge / max_edge if max_edge > 0 else 0.5
    
    position_fraction = MIN_POSITION + (MAX_POSITION - MIN_POSITION) * (normalized ** SIZE_EXPONENT)
    position_fraction = np.clip(position_fraction * size_multiplier, MIN_POSITION, MAX_POSITION)
    
    position_value = capital * position_fraction
    entry_price = current_price
    max_price = current_price
    
    # ==================== DYNAMIC ATR-BASED STOPS ====================
    # Adjust stops based on current volatility
    vol_adjusted_stop = BASE_STOP_LOSS + (current_atr / current_price) * 0.5
    vol_adjusted_stop = np.clip(vol_adjusted_stop, BASE_STOP_LOSS, BASE_STOP_LOSS * 1.5)
    
    vol_adjusted_profit = BASE_TAKE_PROFIT * (0.8 + vol_regime * 0.4)  # Tighter TP in high vol
    
    # ==================== EXIT LOGIC ====================
    exit_price = prices[i + HORIZON]
    exit_idx = i + HORIZON
    exit_reason = "TIME"
    
    for j in range(1, HORIZON + 1):
        price = prices[i + j]
        max_price = max(max_price, price)
        
        # Dynamic stop loss
        if price <= entry_price * (1 - vol_adjusted_stop):
            exit_price = entry_price * (1 - vol_adjusted_stop)
            exit_idx = i + j
            exit_reason = "STOP"
            break
        
        # Dynamic take profit
        if price >= entry_price * (1 + vol_adjusted_profit):
            exit_price = entry_price * (1 + vol_adjusted_profit)
            exit_idx = i + j
            exit_reason = "PROFIT"
            break
        
        # Trailing stop
        trail_level = max_price * (1 - TRAIL_STOP)
        if price < trail_level:
            exit_price = trail_level
            exit_idx = i + j
            exit_reason = "TRAIL"
            break
    
    # ==================== PnL ====================
    ret = (exit_price - entry_price) / entry_price
    net_ret = ret - (COST_PER_TRADE * 2)
    
    pnl = position_value * net_ret
    capital += pnl
    
    if capital < 0:
        capital = INITIAL_CAPITAL * 0.001
    
    # Track for adaptive sizing
    recent_returns.append(net_ret)
    
    trades.append({
        "entry_idx": i,
        "exit_idx": exit_idx,
        "entry_price": entry_price,
        "exit_price": exit_price,
        "prob": prob,
        "size": position_fraction,
        "return": net_ret,
        "pnl": pnl,
        "capital": capital,
        "exit_reason": exit_reason,
        "vol": test_df["volatility_10"].iloc[i],
        "trend_strength": test_df["trend_strength"].iloc[i]
    })
    
    equity_curve.append(capital)
    last_exit = exit_idx
    i = exit_idx + COOLDOWN

print(f"\n✅ Advanced Backtest complete: {len(trades)} trades")
print(f"💰 Capital: {INITIAL_CAPITAL:,.0f} → {capital:,.0f}")



📊 ML Ensemble Probability Distribution:
  Min: 0.1085
  25%: 0.2436
  Median: 0.2966
  75%: 0.3439
  95%: 0.4128
  Max: 0.5387

📊 Entry Threshold (Q=0.14): 0.2122
📊 Potential signals: 69580 / 80907

✅ Advanced Backtest complete: 3994 trades
💰 Capital: 1,000,000 → 1,167,840


In [10]:

# ======================================================
# RETRAIN MODEL WITH PATH 1 PARAMS FOR BACKTEST
# ======================================================
print("🔄 Retraining XGBoost with PATH 1 (Reduced Regularization)...")

from xgboost import XGBClassifier

# Prepare features
feature_cols = [
    col for col in train_df.columns
    if col not in ['target', 'Open', 'High', 'Low', 'Close', 'Volume', 'strategy_signal']
]

X_train_bt = train_df[feature_cols]
y_train_bt = train_df['target']
X_test_bt = test_df[feature_cols]

# Scale
from sklearn.preprocessing import StandardScaler
scaler_bt = StandardScaler()
X_train_scaled_bt = scaler_bt.fit_transform(X_train_bt)
X_test_scaled_bt = scaler_bt.transform(X_test_bt)

# Train with PATH 1 parameters
scale_pos_weight_bt = (len(X_train_scaled_bt) - y_train_bt.sum()) / max(y_train_bt.sum(), 1)

model_bt = XGBClassifier(
    n_estimators=200,
    max_depth=4,            # ✅ PATH 1: INCREASED from 3
    learning_rate=0.02,     # ✅ PATH 1: INCREASED from 0.01
    subsample=0.7,          # ✅ PATH 1: INCREASED from 0.5
    colsample_bytree=0.7,   # ✅ PATH 1: INCREASED from 0.5
    min_child_weight=10,    # ✅ PATH 1: REDUCED from 50
    gamma=0.1,              # ✅ PATH 1: REDUCED from 0.5
    reg_alpha=0.1,          # ✅ PATH 1: CRITICAL CHANGE (from 0.5)
    reg_lambda=0.5,         # ✅ PATH 1: CRITICAL CHANGE (from 3.0)
    scale_pos_weight=scale_pos_weight_bt,
    objective="binary:logistic",
    eval_metric="auc",
    random_state=42,
    verbosity=0
)

model_bt.fit(X_train_scaled_bt, y_train_bt)

# Get ml_prob for backtest
ml_prob = model_bt.predict_proba(X_test_scaled_bt)[:, 1]

print(f"\n✅ PATH 1 MODEL READY FOR BACKTEST")
print(f"   Probability distribution:")
print(f"   Min: {ml_prob.min():.4f}, Max: {ml_prob.max():.4f}")
print(f"   Mean: {ml_prob.mean():.4f}, Median: {np.median(ml_prob):.4f}")
print(f"   THIS SHOULD BE MUCH BETTER THAN BEFORE (max was 0.533)")


🔄 Retraining XGBoost with PATH 1 (Reduced Regularization)...

✅ PATH 1 MODEL READY FOR BACKTEST
   Probability distribution:
   Min: 0.2442, Max: 0.6661
   Mean: 0.4661, Median: 0.4739
   THIS SHOULD BE MUCH BETTER THAN BEFORE (max was 0.533)


In [11]:
equity = pd.Series(equity_curve)
returns = equity.pct_change().dropna()

total_return = (equity.iloc[-1] / equity.iloc[0]) - 1
max_dd = ((equity / equity.cummax()) - 1).min()

# Proper intraday annualization
sharpe = (
    returns.mean() / returns.std()
    if returns.std() > 0 else 0
) * np.sqrt(252 * 6.5 * 60)

# Profit factor calculation
winning_trades = [t["pnl"] for t in trades if t["pnl"] > 0]
losing_trades = [t["pnl"] for t in trades if t["pnl"] < 0]

gross_profit = sum(winning_trades) if winning_trades else 0
gross_loss = abs(sum(losing_trades)) if losing_trades else 1e-8

profit_factor = gross_profit / gross_loss if gross_loss > 0 else np.inf

win_rate = len(winning_trades) / len(trades) if trades else 0

# Expectancy calculation
avg_win = np.mean(winning_trades) if winning_trades else 0
avg_loss = np.mean(losing_trades) if losing_trades else 0
expectancy = (win_rate * avg_win) + ((1 - win_rate) * avg_loss)

print("\n" + "="*60)
print("🎯 BACKTEST RESULTS (OPTIMIZED FOR AUC 0.60 EDGE)")
print("="*60)
print(f"Initial Capital:   ₹{equity.iloc[0]:,.0f}")
print(f"Final Capital:     ₹{equity.iloc[-1]:,.0f}")
print(f"Total Return:      {total_return*100:.2f}%")
print(f"Net Profit:        ₹{equity.iloc[-1] - equity.iloc[0]:,.0f}")
print(f"Max Drawdown:      {max_dd*100:.2f}%")
print(f"Sharpe Ratio:      {sharpe:.2f}")
print(f"Profit Factor:     {profit_factor:.2f}")
print(f"Win Rate:          {win_rate*100:.2f}%")
print(f"Avg Win:           ₹{avg_win:,.0f}")
print(f"Avg Loss:          ₹{avg_loss:,.0f}")
print(f"Expectancy:        ₹{expectancy:,.0f} per trade")
print(f"Total Trades:      {len(trades)}")
print(f"Cost Per Trade:    {COST_PER_TRADE*100:.4f}% per side")
print("="*60)

# ==================== DIAGNOSTIC ANALYSIS ====================
print("\n" + "="*60)
print("🔍 DIAGNOSTIC REPORT")
print("="*60)

if len(trades) < 100:
    print(f"⚠️  WARNING: Only {len(trades)} trades - sample size too small")
    print("   Need 200+ trades for weak edge (AUC 0.60) to show in backtest")

# Check if cost is the problem
total_costs_paid = len(trades) * COST_PER_TRADE * 2 * INITIAL_CAPITAL / 1_000_000
print(f"📊 Total costs paid (estimate): ₹{total_costs_paid:,.0f}")
print(f"📊 Average profit needed per trade: ₹{total_return * INITIAL_CAPITAL / len(trades):,.0f}" if trades else "N/A")

if total_return <= 0 and len(trades) > 50:
    print("\n❌ STILL LOSING - Multiple issues:")
    
    if len(trades) < 150:
        print(f"   1. Volume too low ({len(trades)} trades)")
        print("      → Lower ENTRY_Q further (try 0.70)")
    
    if profit_factor < 1.0:
        print(f"   2. Win rate too low ({win_rate*100:.1f}%)")
        print("      → Increase STOP_LOSS (try 0.020)")
        print("      → Widen TAKE_PROFIT (try 0.030)")
    
    if avg_win <= abs(avg_loss):
        print(f"   3. Risk/reward imbalanced ({avg_win:.0f} win vs {avg_loss:.0f} loss)")
        print("      → Your model edge is weaker than expected")
        print("      → Consider retraining with different target")

elif total_return > 0:
    print(f"\n✅ PROFITABLE! +{total_return*100:.2f}% return on {len(trades)} trades")
    if profit_factor > 1.5:
        print("   Strategy is working - edge is clear")
    elif profit_factor > 1.0:
        print("   Strategy works but edge is small - scale up carefully")
else:
    print("\n⚠️  MARGINALLY PROFITABLE")
    print("   Continue testing with more data")



🎯 BACKTEST RESULTS (OPTIMIZED FOR AUC 0.60 EDGE)
Initial Capital:   ₹1,000,000
Final Capital:     ₹1,167,840
Total Return:      16.78%
Net Profit:        ₹167,840
Max Drawdown:      -4.73%
Sharpe Ratio:      3.29
Profit Factor:     1.12
Win Rate:          49.77%
Avg Win:           ₹760
Avg Loss:          ₹-669
Expectancy:        ₹42 per trade
Total Trades:      3994
Cost Per Trade:    0.0001% per side

🔍 DIAGNOSTIC REPORT
📊 Total costs paid (estimate): ₹0
📊 Average profit needed per trade: ₹42

✅ PROFITABLE! +16.78% return on 3994 trades
   Strategy works but edge is small - scale up carefully


In [12]:

# ======================================================
# CRITICAL: DIAGNOSE ML MODEL CALIBRATION
# ======================================================
print("\n" + "="*70)
print("🚨 ML MODEL CALIBRATION ANALYSIS")
print("="*70)

# Check actual trade distribution
trade_returns = [t["return"] for t in trades]

print(f"\n1️⃣  PROBABILITY DISTRIBUTION (Your model outputs):")
print(f"   Min: {ml_prob.min():.4f}")
print(f"   Max: {ml_prob.max():.4f}")
print(f"   Mean: {ml_prob.mean():.4f}")
print(f"   Median: {np.median(ml_prob):.4f}")
print(f"   Signals above 0.50 (neutral): {(ml_prob > 0.5).sum()} / {len(ml_prob)}")
print(f"   Signals above 0.60: {(ml_prob > 0.6).sum()} / {len(ml_prob)}")

if ml_prob.max() < 0.60:
    print("\n   ❌ PROBLEM: Model never outputs >60% confidence!")
    print("      This suggests the model is UNDERFITTED or REGULARIZED TOO HEAVILY")
    print("      → Check: max_depth, n_estimators, reg_alpha, reg_lambda")

if ml_prob.mean() < 0.40:
    print("\n   ⚠️  WARNING: Average confidence is {:.2%}".format(ml_prob.mean()))
    print("      Model probabilities are CALIBRATED to SELL signals (0 class)")
    print("      The target may be: does price go DOWN, not UP")

print(f"\n2️⃣  CLASS DISTRIBUTION (Training data):")
print(f"   Target 0 (no move): {(y_test == 0).sum()} samples")
print(f"   Target 1 (upside): {(y_test == 1).sum()} samples")
print(f"   Class ratio: {(y_test == 0).sum() / (y_test == 1).sum():.1f}:1")

if (y_test == 0).sum() / (y_test == 1).sum() > 10:
    print("   ❌ SEVERE CLASS IMBALANCE: >90% of samples are NEGATIVE class")
    print("      Model learns to predict 0 by default → all outputs cluster near 0")

print(f"\n3️⃣  TRADING RESULTS vs TARGET:")
print(f"   Actual trades executed: {len(trades)}")
print(f"   Win rate: {np.mean([1 for t in trades if t['return'] > 0]):.1%}")
print(f"   Expected win rate (for AUC 0.60): ~55%")

if np.mean([1 for t in trades if t['return'] > 0]) < 0.45:
    print("\n   ❌ WIN RATE BELOW 45%: Model predicts WORSE than random")
    print("      This is CONSISTENT with low probability outputs")
    print("      You're trading signals the model says are WEAK")

print(f"\n4️⃣  SOLUTION:")
print("   • Try using INVERSE predictions: entry when prob < 0.35")
print("   • Or retrain model with:")
print("      - Different target variable")
print("      - Less regularization (lower reg_alpha/reg_lambda)")
print("      - Different horizon (try 1, 5, 10, 20 bar horizons)")
print("="*70)

if not trades:
    print("\n❌ NO TRADES EXECUTED - Model is completely uncalibrated")



🚨 ML MODEL CALIBRATION ANALYSIS

1️⃣  PROBABILITY DISTRIBUTION (Your model outputs):
   Min: 0.2442
   Max: 0.6661
   Mean: 0.4661
   Median: 0.4739
   Signals above 0.50 (neutral): 30811 / 80907
   Signals above 0.60: 2817 / 80907

2️⃣  CLASS DISTRIBUTION (Training data):
   Target 0 (no move): 58125 samples
   Target 1 (upside): 22782 samples
   Class ratio: 2.6:1

3️⃣  TRADING RESULTS vs TARGET:
   Actual trades executed: 3994
   Win rate: 100.0%
   Expected win rate (for AUC 0.60): ~55%

4️⃣  SOLUTION:
   • Try using INVERSE predictions: entry when prob < 0.35
   • Or retrain model with:
      - Different target variable
      - Less regularization (lower reg_alpha/reg_lambda)
      - Different horizon (try 1, 5, 10, 20 bar horizons)


In [16]:

# ======================================================
# LIVE MARKET SIMULATION - AGGRESSIVE PROFIT OPTIMIZATION
# ======================================================
import yfinance as yf
from datetime import datetime, timedelta

# ============= ULTRA-AGGRESSIVE PARAMETERS =============
LIVE_ENTRY_THRESHOLD = 0.10  # ✅ VERY LOW - catch all strong signals
LIVE_ENTRY_Q = 0.40          # ✅ LARGE position size - max 40% per trade
LIVE_STOP_LOSS = 0.015       # ✅ TIGHT - 1.5% stop (preserve capital)
LIVE_TAKE_PROFIT = 0.025     # ✅ TIGHT - 2.5% profit target (lock gains fast)

print("\n" + "="*80)
print("🚀 LIVE MARKET SIMULATION - AGGRESSIVE PROFIT MODE")
print("="*80)
print(f"\n⚙️  ULTRA-AGGRESSIVE PARAMETERS:")
print(f"   Entry Threshold: {LIVE_ENTRY_THRESHOLD:.4f} (was 0.12 - MAXIMUM signals)")
print(f"   Position Size: {LIVE_ENTRY_Q:.2f} (was 0.25 - BIGGEST positions)")
print(f"   Stop Loss: {LIVE_STOP_LOSS*100:.2f}% (was 1.80% - TIGHTEST)")
print(f"   Take Profit: {LIVE_TAKE_PROFIT*100:.2f}% (was 3.50% - FASTEST close)")

# Fetch recent 1 day data from yfinance
ticker_to_test = "^NSEBANK"  # NIFTY BANK
end_date = datetime.now()
start_date = end_date - timedelta(days=5)  # 5 days of data for features

print(f"\n📊 Fetching {ticker_to_test} data from yfinance...")
print(f"   Period: {start_date.date()} to {end_date.date()}")

try:
    yf_data = yf.download(ticker_to_test, start=start_date, end=end_date, interval="1m", progress=False)
    
    # Flatten multiindex columns if they exist
    if isinstance(yf_data.columns, pd.MultiIndex):
        yf_data.columns = [col[0] for col in yf_data.columns]
    
    # Select only OHLCV columns
    yf_data = yf_data[['Open', 'High', 'Low', 'Close', 'Volume']].copy()
    
    print(f"✅ Downloaded {len(yf_data)} minute candles")
    print(f"   Date range: {yf_data.index[0]} to {yf_data.index[-1]}")
    print(f"   Price range: ₹{float(yf_data['Close'].min()):.2f} - ₹{float(yf_data['Close'].max()):.2f}")
    
except Exception as e:
    print(f"❌ Error fetching data: {e}")
    import traceback
    traceback.print_exc()
    yf_data = None

if yf_data is not None and len(yf_data) > 50:
    # Clean data
    yf_clean = yf_data.dropna()
    
    print(f"\n🧹 After cleaning: {len(yf_clean)} valid candles")
    
    if len(yf_clean) > 0:
        # Process data through the exact same pipeline as training
        # Step 1: Add scalping signals
        live_with_signals = add_scalping_signals(yf_clean.copy())
        
        # Step 2: Add basic features
        live_with_basic = add_basic_features(live_with_signals.copy())
        
        # Step 3: Add advanced features (multi-timeframe)
        live_with_all_features = add_advanced_features(live_with_basic.copy())
        
        # Flatten column names if multiindex
        if isinstance(live_with_all_features.columns, pd.MultiIndex):
            live_with_all_features.columns = [col[0] if isinstance(col, tuple) else col for col in live_with_all_features.columns]
        
        # Clean NaN values
        live_features_clean = live_with_all_features.dropna()
        
        print(f"   Valid rows after features: {len(live_features_clean)}")
        
        # Get the exact feature columns used in training
        feature_cols_live = [
            c for c in live_features_clean.columns
            if c not in ["target", "Open", "High", "Low", "Close", "Volume", "strategy_signal", "tr", "high_20", "low_20"]
        ]
        
        print(f"   Features to use: {len(feature_cols_live)}")
        
        if len(live_features_clean) > 0 and len(feature_cols_live) > 0:
            X_live = live_features_clean[feature_cols_live].copy()
            X_live_scaled = scaler.transform(X_live)
            
            # Get predictions from both models
            xgb_pred_live = xgb_model.predict_proba(X_live_scaled)[:, 1]
            lgb_pred_live = lgb_model.predict_proba(X_live_scaled)[:, 1]
            ensemble_pred_live = (xgb_pred_live * 0.55 + lgb_pred_live * 0.45)  # Same blend as training
            
            print(f"\n🤖 Model Predictions on recent data:")
            print(f"   XGBoost  - Min: {float(xgb_pred_live.min()):.4f}, Max: {float(xgb_pred_live.max()):.4f}, Mean: {float(xgb_pred_live.mean()):.4f}")
            print(f"   LightGBM - Min: {float(lgb_pred_live.min()):.4f}, Max: {float(lgb_pred_live.max()):.4f}, Mean: {float(lgb_pred_live.mean()):.4f}")
            print(f"   Ensemble - Min: {float(ensemble_pred_live.min()):.4f}, Max: {float(ensemble_pred_live.max()):.4f}, Mean: {float(ensemble_pred_live.mean()):.4f}")
            
            # Simulate trading
            capital_live = float(INITIAL_CAPITAL)
            positions_live = []
            pnl_live = 0.0
            entry_count = 0
            exit_count = 0
            profitable_trades = 0
            losing_trades = 0
            max_capital = capital_live
            trade_log = []
            
            print(f"\n💰 Starting Capital: ₹{capital_live:,.0f}")
            print(f"\n{'📍 AGGRESSIVE TRADING SESSION':^70}")
            
            # Simulate real-time trading
            for i in range(len(live_features_clean)):
                current_data = live_features_clean.iloc[i]
                current_price = float(current_data['Close'])
                current_time = live_features_clean.index[i]
                
                signal = int(current_data['strategy_signal'])
                ensemble_score = float(ensemble_pred_live[i])
                
                # Check for exits first
                closed_indices = []
                for j, pos in enumerate(positions_live):
                    price_move = (current_price - pos['entry_price']) / pos['entry_price']
                    
                    if price_move >= LIVE_TAKE_PROFIT or price_move <= -LIVE_STOP_LOSS:
                        # Exit the position
                        exit_cost = pos['qty'] * current_price * float(COST_PER_TRADE)
                        exit_pnl = pos['qty'] * (current_price - pos['entry_price']) - (pos['entry_cost'] + exit_cost)
                        capital_live += exit_pnl
                        pnl_live += exit_pnl
                        max_capital = max(max_capital, capital_live)
                        exit_count += 1
                        
                        if exit_pnl > 0:
                            profitable_trades += 1
                        else:
                            losing_trades += 1
                        
                        closed_indices.append(j)
                        
                        exit_reason = "🎯 PROFIT" if exit_pnl > 0 else "🛑 STOP"
                        print(f"\n{exit_reason} EXIT at {current_time} | Price: ₹{current_price:.2f} | P&L: ₹{exit_pnl:,.0f}")
                        
                        trade_log.append({
                            'entry': pos['entry_price'],
                            'exit': current_price,
                            'pnl': exit_pnl,
                            'reason': exit_reason
                        })
                        
                # Remove closed positions
                positions_live = [pos for idx, pos in enumerate(positions_live) if idx not in closed_indices]
                
                # Check for entries - with ultra-aggressive threshold
                if signal == 1 and ensemble_score > LIVE_ENTRY_THRESHOLD and len(positions_live) == 0:
                    entry_cost = capital_live * LIVE_ENTRY_Q * float(COST_PER_TRADE)
                    qty = int(capital_live * LIVE_ENTRY_Q / current_price)
                    
                    if qty > 0:
                        entry_count += 1
                        positions_live.append({
                            'entry_price': current_price,
                            'qty': qty,
                            'entry_cost': entry_cost,
                            'entry_time': current_time
                        })
                        
                        print(f"\n🟢 ENTRY #{entry_count} at {current_time} | Price: ₹{current_price:.2f} | Score: {ensemble_score:.4f}")
            
            # Close any remaining positions at market
            for pos in positions_live:
                last_price = float(live_features_clean['Close'].iloc[-1])
                exit_cost = pos['qty'] * last_price * float(COST_PER_TRADE)
                exit_pnl = pos['qty'] * (last_price - pos['entry_price']) - (pos['entry_cost'] + exit_cost)
                capital_live += exit_pnl
                pnl_live += exit_pnl
                max_capital = max(max_capital, capital_live)
                exit_count += 1
                
                if exit_pnl > 0:
                    profitable_trades += 1
                else:
                    losing_trades += 1
                
                print(f"\n⏹️  MARKET CLOSE | Price: ₹{last_price:.2f} | Final P&L: ₹{exit_pnl:,.0f}")
                
                trade_log.append({
                    'entry': pos['entry_price'],
                    'exit': last_price,
                    'pnl': exit_pnl,
                    'reason': 'MARKET_CLOSE'
                })
            
            final_return = (capital_live - float(INITIAL_CAPITAL)) / float(INITIAL_CAPITAL)
            win_rate = profitable_trades / (profitable_trades + losing_trades) if (profitable_trades + losing_trades) > 0 else 0
            avg_profit = pnl_live / entry_count if entry_count > 0 else 0
            
            print(f"\n" + "="*80)
            print(f"📊 AGGRESSIVE MODE RESULTS - {ticker_to_test}")
            print(f"="*80)
            print(f"Initial Capital:    ₹{float(INITIAL_CAPITAL):,.0f}")
            print(f"Final Capital:      ₹{capital_live:,.0f}")
            print(f"Peak Capital:       ₹{max_capital:,.0f}")
            print(f"Total P&L:          ₹{pnl_live:,.0f}")
            print(f"Return %:           {final_return*100:+.2f}%")
            print(f"---")
            print(f"Total Entries:      {entry_count}")
            print(f"Profitable Trades:  {profitable_trades}")
            print(f"Losing Trades:      {losing_trades}")
            print(f"Win Rate:           {win_rate*100:.1f}%")
            print(f"Avg P&L per Trade:  ₹{avg_profit:,.0f}")
            print(f"="*80)
            
            if final_return > 0:
                improvement_from_first = ((capital_live - 1002388) / 1002388) * 100 if capital_live > INITIAL_CAPITAL else 0
                print(f"\n✅ PROFITABLE! Return: {final_return*100:+.2f}%")
                if improvement_from_first > 0:
                    print(f"   📈 Total improvement from +0.24%: {improvement_from_first:.2f}%")
                print("🔥 Aggressive strategy is DOMINATING!")
            else:
                print(f"\n❌ NOT PROFITABLE. Return: {final_return*100:.2f}%")
                print("📝 Rerun with current parameters or adjust further...")
                
        else:
            print(f"❌ Not enough valid data!")
    else:
        print(f"❌ No data after cleaning")
else:
    print("❌ Failed to fetch sufficient data from yfinance")



🚀 LIVE MARKET SIMULATION - AGGRESSIVE PROFIT MODE

⚙️  ULTRA-AGGRESSIVE PARAMETERS:
   Entry Threshold: 0.1000 (was 0.12 - MAXIMUM signals)
   Position Size: 0.40 (was 0.25 - BIGGEST positions)
   Stop Loss: 1.50% (was 1.80% - TIGHTEST)
   Take Profit: 2.50% (was 3.50% - FASTEST close)

📊 Fetching ^NSEBANK data from yfinance...
   Period: 2025-12-29 to 2026-01-03
✅ Downloaded 1622 minute candles
   Date range: 2025-12-29 07:58:00+00:00 to 2026-01-02 09:59:00+00:00
   Price range: ₹58746.90 - ₹60188.05

🧹 After cleaning: 1622 valid candles
   Valid rows after features: 1543
   Features to use: 25

🤖 Model Predictions on recent data:
   XGBoost  - Min: 0.1136, Max: 0.3901, Mean: 0.2033
   LightGBM - Min: 0.1170, Max: 0.4035, Mean: 0.2061
   Ensemble - Min: 0.1152, Max: 0.3961, Mean: 0.2045

💰 Starting Capital: ₹1,000,000

                     📍 AGGRESSIVE TRADING SESSION                     

🟢 ENTRY #1 at 2025-12-29 09:27:00+00:00 | Price: ₹58958.75 | Score: 0.2078

⏹️  MARKET CLOSE | 

In [ ]:

# ======================================================
# LIVE PAPER TRADING - CONTINUOUS REAL-TIME TRADING
# ======================================================
import yfinance as yf
from datetime import datetime, timedelta
import time
import json
from pathlib import Path

# ============= PAPER TRADING CONFIG =============
INITIAL_PAPER_CAPITAL = 1_000_000
PAPER_ENTRY_THRESHOLD = 0.10
PAPER_ENTRY_Q = 0.40
PAPER_STOP_LOSS = 0.015
PAPER_TAKE_PROFIT = 0.025
CHECK_INTERVAL = 5  # Check every 5 seconds (for testing; use 60 for 1-min candles)
MAX_RUNTIME = 3600  # 1 hour max for testing (change to 25200 for 7 hours trading day)

print("\n" + "="*80)
print("📈 LIVE PAPER TRADING SYSTEM")
print("="*80)
print(f"\n⚙️  PAPER TRADING CONFIGURATION:")
print(f"   Initial Capital: ₹{INITIAL_PAPER_CAPITAL:,.0f}")
print(f"   Entry Threshold: {PAPER_ENTRY_THRESHOLD:.4f}")
print(f"   Position Size: {PAPER_ENTRY_Q:.0%}")
print(f"   Stop Loss: {PAPER_STOP_LOSS*100:.2f}%")
print(f"   Take Profit: {PAPER_TAKE_PROFIT*100:.2f}%")
print(f"   Check Interval: {CHECK_INTERVAL}s")
print(f"   Max Runtime: {MAX_RUNTIME}s ({MAX_RUNTIME/60:.0f} minutes)")
print("="*80)

# Initialize paper trading state
paper_trading_state = {
    'capital': INITIAL_PAPER_CAPITAL,
    'peak_capital': INITIAL_PAPER_CAPITAL,
    'positions': [],
    'completed_trades': [],
    'pnl': 0.0,
    'start_time': datetime.now(),
    'last_candle_time': None,
    'candles_processed': 0
}

# Trading log file
log_file = Path("paper_trading_log.json")
if log_file.exists():
    with open(log_file, 'r') as f:
        paper_trading_state = json.load(f)
        # Reset dates for new session
        paper_trading_state['start_time'] = datetime.now().isoformat()
        paper_trading_state['positions'] = []
        paper_trading_state['completed_trades'] = []
        paper_trading_state['candles_processed'] = 0

def save_trading_state():
    """Save current trading state to file"""
    state_copy = paper_trading_state.copy()
    state_copy['start_time'] = paper_trading_state['start_time'].isoformat() if isinstance(paper_trading_state['start_time'], datetime) else paper_trading_state['start_time']
    with open(log_file, 'w') as f:
        json.dump(state_copy, f, indent=2, default=str)

def fetch_live_data(ticker, lookback_days=5):
    """Fetch latest OHLCV data"""
    try:
        end_date = datetime.now()
        start_date = end_date - timedelta(days=lookback_days)
        
        data = yf.download(ticker, start=start_date, end=end_date, interval="1m", progress=False)
        
        # Clean columns
        if isinstance(data.columns, pd.MultiIndex):
            data.columns = [col[0] for col in data.columns]
        
        data = data[['Open', 'High', 'Low', 'Close', 'Volume']].dropna()
        return data
    except Exception as e:
        print(f"❌ Error fetching data: {e}")
        return None

def process_live_signals(data):
    """Process data and generate trading signals"""
    try:
        # Apply feature pipeline
        with_signals = add_scalping_signals(data.copy())
        with_basic = add_basic_features(with_signals.copy())
        with_all = add_advanced_features(with_basic.copy())
        
        # Clean
        clean_data = with_all.dropna()
        
        # Get features
        feature_cols_live = [
            c for c in clean_data.columns
            if c not in ["target", "Open", "High", "Low", "Close", "Volume", "strategy_signal", "tr", "high_20", "low_20"]
        ]
        
        # Get latest predictions
        X_live = clean_data[feature_cols_live].copy()
        X_live_scaled = scaler.transform(X_live)
        
        xgb_pred = xgb_model.predict_proba(X_live_scaled)[:, 1]
        lgb_pred = lgb_model.predict_proba(X_live_scaled)[:, 1]
        ensemble_pred = (xgb_pred * 0.55 + lgb_pred * 0.45)
        
        return clean_data, ensemble_pred, feature_cols_live
    except Exception as e:
        print(f"❌ Error processing signals: {e}")
        return None, None, None

def update_positions(current_price, current_time):
    """Check existing positions for exits"""
    closed = []
    
    for idx, pos in enumerate(paper_trading_state['positions']):
        price_move = (current_price - pos['entry_price']) / pos['entry_price']
        
        # Check exit conditions
        if price_move >= PAPER_TAKE_PROFIT or price_move <= -PAPER_STOP_LOSS:
            # Calculate exit P&L
            exit_cost = pos['qty'] * current_price * float(COST_PER_TRADE)
            exit_pnl = pos['qty'] * (current_price - pos['entry_price']) - (pos['entry_cost'] + exit_cost)
            
            # Update capital and state
            paper_trading_state['capital'] += exit_pnl
            paper_trading_state['pnl'] += exit_pnl
            paper_trading_state['peak_capital'] = max(paper_trading_state['peak_capital'], paper_trading_state['capital'])
            
            # Log completed trade
            exit_reason = "PROFIT" if exit_pnl > 0 else "STOP"
            paper_trading_state['completed_trades'].append({
                'entry_time': pos['entry_time'],
                'exit_time': current_time,
                'entry_price': pos['entry_price'],
                'exit_price': current_price,
                'qty': pos['qty'],
                'pnl': exit_pnl,
                'reason': exit_reason,
                'return_pct': price_move * 100
            })
            
            closed.append(idx)
            emoji = "🎯" if exit_pnl > 0 else "🛑"
            print(f"\n{emoji} EXIT | Time: {current_time} | Price: ₹{current_price:.2f} | P&L: ₹{exit_pnl:,.0f} ({price_move*100:+.2f}%)")
    
    # Remove closed positions
    paper_trading_state['positions'] = [pos for i, pos in enumerate(paper_trading_state['positions']) if i not in closed]
    
    return len(closed)

def check_entry_signals(clean_data, ensemble_pred, current_price, current_time):
    """Check for new entry signals"""
    if len(paper_trading_state['positions']) > 0:
        return 0  # Only one position at a time
    
    # Get latest signal and score
    latest_signal = int(clean_data['strategy_signal'].iloc[-1])
    latest_score = float(ensemble_pred[-1])
    
    if latest_signal == 1 and latest_score > PAPER_ENTRY_THRESHOLD:
        # Calculate position size
        entry_cost = paper_trading_state['capital'] * PAPER_ENTRY_Q * float(COST_PER_TRADE)
        qty = int(paper_trading_state['capital'] * PAPER_ENTRY_Q / current_price)
        
        if qty > 0:
            # Enter position
            paper_trading_state['positions'].append({
                'entry_price': current_price,
                'entry_time': current_time,
                'qty': qty,
                'entry_cost': entry_cost,
                'entry_score': latest_score
            })
            
            print(f"\n🟢 ENTRY | Time: {current_time} | Price: ₹{current_price:.2f} | Score: {latest_score:.4f} | Size: {PAPER_ENTRY_Q:.0%}")
            return 1
    
    return 0

def print_status():
    """Print current trading status"""
    current_return = (paper_trading_state['capital'] - INITIAL_PAPER_CAPITAL) / INITIAL_PAPER_CAPITAL
    
    print(f"\n{'─'*80}")
    print(f"💰 CAPITAL: ₹{paper_trading_state['capital']:,.0f} | P&L: ₹{paper_trading_state['pnl']:,.0f} | Return: {current_return*100:+.2f}%")
    print(f"📊 Candles: {paper_trading_state['candles_processed']} | Open Positions: {len(paper_trading_state['positions'])} | Closed: {len(paper_trading_state['completed_trades'])}")
    
    if paper_trading_state['completed_trades']:
        wins = sum(1 for t in paper_trading_state['completed_trades'] if t['pnl'] > 0)
        losses = len(paper_trading_state['completed_trades']) - wins
        win_rate = wins / len(paper_trading_state['completed_trades']) * 100
        print(f"📈 Win Rate: {win_rate:.1f}% ({wins}W/{losses}L)")

# ============= LIVE TRADING LOOP =============
print(f"\n🚀 Starting Live Paper Trading Session at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"{'─'*80}\n")

start_time = time.time()
iteration = 0

while (time.time() - start_time) < MAX_RUNTIME:
    try:
        iteration += 1
        current_time = datetime.now()
        
        # Fetch latest data
        raw_data = fetch_live_data("^NSEBANK", lookback_days=5)
        
        if raw_data is not None and len(raw_data) > 50:
            clean_data, ensemble_pred, feature_cols = process_live_signals(raw_data)
            
            if clean_data is not None and ensemble_pred is not None:
                current_price = float(clean_data['Close'].iloc[-1])
                
                # Update existing positions (check for exits)
                exits = update_positions(current_price, current_time.strftime('%H:%M:%S'))
                
                # Check for new entries
                entries = check_entry_signals(clean_data, ensemble_pred, current_price, current_time)
                
                paper_trading_state['candles_processed'] += 1
                paper_trading_state['last_candle_time'] = current_time.isoformat()
                
                # Print status every 10 iterations
                if iteration % 10 == 0:
                    print_status()
        
        # Save state every iteration
        save_trading_state()
        
        # Wait before next check
        time.sleep(CHECK_INTERVAL)
        
    except KeyboardInterrupt:
        print("\n\n⏸️  Paper trading paused by user")
        break
    except Exception as e:
        print(f"⚠️  Error in trading loop: {e}")
        time.sleep(CHECK_INTERVAL * 2)
        continue

# ============= FINAL REPORT =============
print("\n\n" + "="*80)
print("📊 LIVE PAPER TRADING SESSION COMPLETE")
print("="*80)

final_return = (paper_trading_state['capital'] - INITIAL_PAPER_CAPITAL) / INITIAL_PAPER_CAPITAL
session_duration = (time.time() - start_time) / 60

print(f"\n⏱️  Session Duration: {session_duration:.1f} minutes")
print(f"💰 Initial Capital: ₹{INITIAL_PAPER_CAPITAL:,.0f}")
print(f"💰 Final Capital: ₹{paper_trading_state['capital']:,.0f}")
print(f"💰 Peak Capital: ₹{paper_trading_state['peak_capital']:,.0f}")
print(f"💰 Total P&L: ₹{paper_trading_state['pnl']:,.0f}")
print(f"📈 Return: {final_return*100:+.2f}%")
print(f"━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print(f"📊 Candles Processed: {paper_trading_state['candles_processed']}")
print(f"🏪 Open Positions: {len(paper_trading_state['positions'])}")
print(f"✅ Completed Trades: {len(paper_trading_state['completed_trades'])}")

if paper_trading_state['completed_trades']:
    wins = sum(1 for t in paper_trading_state['completed_trades'] if t['pnl'] > 0)
    losses = len(paper_trading_state['completed_trades']) - wins
    win_rate = wins / len(paper_trading_state['completed_trades']) * 100
    avg_win = np.mean([t['pnl'] for t in paper_trading_state['completed_trades'] if t['pnl'] > 0]) if wins > 0 else 0
    avg_loss = np.mean([t['pnl'] for t in paper_trading_state['completed_trades'] if t['pnl'] < 0]) if losses > 0 else 0
    
    print(f"🎯 Win Rate: {win_rate:.1f}% ({wins}W/{losses}L)")
    print(f"💵 Avg Win: ₹{avg_win:,.0f}")
    print(f"💸 Avg Loss: ₹{avg_loss:,.0f}")

if paper_trading_state['positions']:
    print(f"\n⚠️  Open Positions at Session End:")
    for pos in paper_trading_state['positions']:
        current_move = ((paper_trading_state['capital'] / INITIAL_PAPER_CAPITAL) - 1) * 100
        print(f"   Entry: ₹{pos['entry_price']:.2f} | Qty: {pos['qty']} | Time: {pos['entry_time']}")

print("\n✅ Trading state saved to: paper_trading_log.json")
print("="*80)

# Display recent trades
if paper_trading_state['completed_trades']:
    print("\n📋 Recent Trades:")
    for trade in paper_trading_state['completed_trades'][-5:]:
        emoji = "✅" if trade['pnl'] > 0 else "❌"
        print(f"{emoji} In: ₹{trade['entry_price']:.2f} | Out: ₹{trade['exit_price']:.2f} | P&L: ₹{trade['pnl']:,.0f} ({trade['return_pct']:+.2f}%) | {trade['reason']}")

